### This notebook introduces the basics of working with geospatial data in Python using the GeoPandas library.

In [ ]:
!pip install geopandas rasterio cartopy matplotlib shapely geodatasets folium mapclassify

## Geopandas

### What is geopandas?
- GeoPandas, as the name suggests, extends the popular data science library pandas by adding support for geospatial data. If you are not familiar with pandas, we recommend taking a quick look at its Getting started documentation before proceeding.
https://pandas.pydata.org/docs/getting_started/index.html#getting-started

- Geopandas main advantage is the oppartunity to store data and the corresponding spatial geometry at the same time. The data is stored in `geopandas.GeoDataFrame`.

- A GeoDataFrame is similar to a pandas DataFrame but includes a special geometry column that stores spatial objects (points, lines, or polygons).

![](https://carpentries-incubator.github.io/geospatial-python/fig/E07/pandas_geopandas_relation.png)

### Imports

In [ ]:
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon

import matplotlib.pyplot as plt
import geopandas
from cartopy import crs as ccrs
import geodatasets
from geodatasets import get_path

### Visualizing

In [ ]:
geodatasets.data

Geospatial data can be stored in various formats such as Shapefile, GeoJSON, or GeoPackage.
GeoPandas provides a convenient function to read these files into a GeoDataFrame.

In [ ]:
path = get_path("naturalearth.land")
df = gpd.read_file(path)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.crs

Visualizing geospatial data helps to better understand spatial patterns and relationships.
GeoPandas provides a simple interface for plotting geometries directly on a map using Matplotlib.


In [ ]:
df.plot()

In [ ]:
# Define the CartoPy CRS object.
crs = ccrs.AzimuthalEquidistant()

# This can be converted into a `proj4` string/dict compatible with GeoPandas
crs_proj4 = crs.proj4_init
df_ae = df.to_crs(crs_proj4)

# Here's what the plot looks like in GeoPandas
df_ae.plot()

In [ ]:
crs_new = ccrs.Geostationary(
    central_longitude=10.0, satellite_height=35785831,
)
new_geometries = [
    crs_new.project_geometry(ii, src_crs=crs) for ii in df_ae["geometry"].values
]

fig, ax = plt.subplots(subplot_kw={"projection": crs_new})
ax.add_geometries(new_geometries, crs=crs_new)

Other projections at https://scitools.org.uk/cartopy/docs/v0.15/crs/projections.html

Visualizations at https://vcg.isti.cnr.it/~tarini/spinnableworldmaps/index.html

In [ ]:
df_aea = geopandas.GeoDataFrame(
    df.drop(columns="geometry"), geometry=new_geometries, crs=crs_new.proj4_init
)
df_aea.plot()

In [ ]:
# Generate a CartoPy figure and add the countries to it
fig, ax = plt.subplots(subplot_kw={"projection": crs_new})
ax.add_geometries(new_geometries, crs=crs_new)

# Calculate centroids and plot
df_aea_centroids = df_aea.geometry.centroid
# Need to provide "zorder" to ensure the points are plotted above the polygons
df_aea_centroids.plot(ax=ax, markersize=5, color="r", zorder=10)
plt.show()

## How to read files

In [ ]:
# gdf = gpd.read_file(path_to_file)
file_location = geodatasets.get_path("nybb")
file_location

In [70]:
gdf = geopandas.read_file(file_location)

In [ ]:
gdf

In [ ]:
gdf.explore()

#### Filter by bbox

In [74]:
bbox = (
    1031051.7879884212, 224272.49231459625, 1047224.3104931959, 244317.30894023244
)
gdf = geopandas.read_file(
    geodatasets.get_path("nybb"),
    bbox=bbox,
)

In [ ]:
gdf

In [ ]:
gdf.explore()

#### Filter by feature

In [78]:
gdf = geopandas.read_file(
    geodatasets.get_path("geoda.nyc"),
    columns=["name", "rent2008", "kids2000"],
)

In [ ]:
gdf = geopandas.read_file(
    geodatasets.get_path("geoda.nyc"),
    where="subborough='Coney Island'",
)

gdf

In [ ]:
gdf.explore()

### Plotting

In [ ]:
chicago = geopandas.read_file(geodatasets.get_path("geoda.chicago_commpop"))

groceries = geopandas.read_file(geodatasets.get_path("geoda.groceries"))

chicago.plot();

In [ ]:
chicago.boundary.plot()

In [ ]:
chicago.plot(
    column="POP2010",
    legend=True,
    legend_kwds={"label": "Population in 2010", "orientation": "horizontal"},
    cmap='OrRd',
);

In [ ]:
groceries.plot(marker='*', color='green', markersize=5);
groceries = groceries.to_crs(chicago.crs)

In [ ]:
base = chicago.plot(color='white', edgecolor='black')
groceries.plot(ax=base, marker='o', color='red', markersize=5);

In [ ]:
import folium

m = chicago.explore(
    column="POP2010",  # make choropleth based on "POP2010" column
    scheme="naturalbreaks",  # use mapclassify's natural breaks scheme
    legend=True,  # show legend
    k=10,  # use 10 bins
    tooltip=False,  # hide tooltip
    popup=["POP2010", "POP2000"],  # show popup (on-click)
    legend_kwds=dict(colorbar=False),  # do not use colorbar
    name="chicago",  # name of the layer in the map
)

groceries.explore(
    m=m,  # pass the map object
    color="red",  # use red color on all points
    marker_kwds=dict(radius=5, fill=True),  # make marker radius 10px with fill
    tooltip="Address",  # show "name" column in the tooltip
    tooltip_kwds=dict(labels=False),  # do not show column label in the tooltip
    name="groceries",  # name of the layer in the map
)

folium.TileLayer("CartoDB positron", show=False).add_to(
    m
)  # use folium to add alternative tiles
folium.LayerControl().add_to(m)  # use folium to add layer control

m  # show map

## How to write

### GeoJSON vs Shapefile

GeoJSON and Shapefile are common formats for storing vector geospatial data.

**GeoJSON** is a text-based format that stores geometry and attributes in a single file.
It is easy to read, share, and use in web applications, but large GeoJSON files can be slow
and inefficient.

**Shapefile** is a widely used GIS format that consists of several files stored together.
It is more compact and often faster than GeoJSON, but less flexible and harder to read
without GIS software.

In general, GeoJSON is better for data exchange and web mapping,
while Shapefile is commonly used in traditional GIS workflows.


In [89]:
from pathlib import Path

Path('data').mkdir(exist_ok=True)

gdf.to_file("data/data.shp")

In [90]:
gdf.to_file("data/data.geojson", driver='GeoJSON')

### Some operations

#### Calculate area

In [91]:
chicago = geopandas.read_file(geodatasets.get_path("geoda.chicago_commpop"))

In [ ]:
chicago.crs

In [ ]:
chicago.area

GeoPandas calculates the area of each polygon using the current CRS (EPSG: 4326).
The resulting values are very small and have no meaningful real-world units because the calculation is performed in degrees squared.

To understand the area of the polygons, we need to reproject the GeoDataFrame to EPSG:3857 (Web Mercator), which is a projected coordinate system that uses metres as units.

These numbers should not be interpreted as the actual area.

In [ ]:
chicago_m = chicago.to_crs('EPSG:3857')
chicago_m.area

#### Check if geomatry containts ROI

In [95]:
p = Point(-87.61868, 41.83512)

In [ ]:
chicago.contains(p).any()

#### Clip the data

In [97]:
from shapely import box


polygon = box(-87.8, 41.90, -87.5, 42)


In [ ]:
chicago_clipped = chicago.clip(polygon)

fig, ax = plt.subplots(figsize=(12, 8))
chicago_clipped.plot(ax=ax)
chicago.boundary.plot(ax=ax)
ax.set_title("Chicago Clipped")
ax.set_axis_off()
plt.show()

In [ ]:
chicago_clipped